[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/ict-gemini-agent-demo/blob/main/ICT_Mini_AI_Agent_Colab.ipynb)

# Python FastAPI와 Gemini API로 구현하는 미니 AI 에이전트

ICT폴리텍대학 교육중점교원 시범강의 실습 노트북 (Google Colab)

이 노트북은 설치 없이 브라우저에서 바로 실행합니다. 위에서부터 셀을 차례로 실행하세요.

학습 흐름: 요청 → 검증 → 프롬프트 → 모델 호출 → 응답 반환

- Part 1. 환경 준비와 API 키 등록
- Part 2. Gemini 모델 직접 호출 (개념 이해)
- Part 3. FastAPI로 웹 API 구조 만들기 (서버 측 호출)
- Part 4. 나만의 학습 도우미 만들기 (과제)

## Part 1. 환경 준비

필요한 패키지를 설치합니다. 약 30초 걸립니다.

In [28]:
!pip install -q google-genai fastapi httpx pydantic

### API 키 등록 (안전한 방법)

API 키는 코드에 직접 적지 않습니다. Colab 왼쪽의 열쇠 모양(보안 비밀) 아이콘을 눌러
이름 `GEMINI_API_KEY`로 키를 등록하고 노트북 액세스를 켭니다.

키 발급: https://aistudio.google.com/apikey

In [29]:
import os

# Colab Secrets에서 키를 읽어 환경변수로 등록 (코드에 키를 노출하지 않음)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API 키 등록 완료')
except Exception as e:
    print('Colab Secrets에서 키를 찾지 못했습니다. 왼쪽 열쇠 아이콘에서 GEMINI_API_KEY를 등록하세요.')
    print('상세:', e)

API 키 등록 완료


In [30]:
from google import genai
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
for m in client.models.list():
    acts = getattr(m, "supported_actions", None) or getattr(m, "supported_generation_methods", [])
    if "generateContent" in (acts or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

## Part 2. Gemini 모델 직접 호출

먼저 모델을 직접 호출해 응답이 오는지 확인합니다. 가장 단순한 형태입니다.

In [31]:
from google import genai
GEMINI_MODEL = 'gemini-2.5-flash'
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

resp = client.models.generate_content(
    model=GEMINI_MODEL,
    contents='AI 에이전트와 일반 챗봇의 차이를 두 문장으로 설명해줘.',
)
print(resp.text)

일반 챗봇은 주로 사용자와 대화하며 정보를 제공하거나 질문에 답하는 데 중점을 둡니다. 반면 AI 에이전트는 사용자의 목표를 이해하고 스스로 판단하여 외부 도구와 연동하며 실제 작업을 계획하고 실행하는 능력을 가집니다.


### 프롬프트를 설계하면 답변 품질이 달라집니다

AI에게 역할, 답변 길이, 예시 조건을 명확히 주는 함수를 만듭니다.

In [33]:
import time
from google.genai import errors


def build_prompt(question: str) -> str:
    return f'''너는 ICT폴리텍 학생을 돕는 AI 학습 도우미다.
아래 질문에 대해 초급 학습자도 이해할 수 있게 5문장 이내로 답하라.
가능하면 실무 예시를 1개 포함하라.

질문: {question}'''


def ask_gemini(question: str, tries: int = 4) -> str:
    """질문을 보내고 답을 받는다. 서버가 혼잡(503)하면 잠시 기다렸다 자동으로 다시 시도한다."""
    for attempt in range(1, tries + 1):
        try:
            resp = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=build_prompt(question),
            )
            return resp.text
        except errors.ServerError:
            if attempt < tries:
                wait = 2 * attempt
                print(f'서버가 혼잡합니다. {wait}초 후 다시 시도합니다... ({attempt}/{tries})')
                time.sleep(wait)
            else:
                return '지금은 서버가 혼잡합니다. 잠시 후 다시 실행해 주세요.'


print(ask_gemini('스마트 컨트랙트가 무엇인가요?'))

안녕하세요, ICT폴리텍 학생 여러분! 스마트 컨트랙트에 대해 쉽게 설명해 드릴게요.

스마트 컨트랙트는 블록체인 위에서 미리 정해진 조건이 충족되면 자동으로 실행되는 디지털 계약입니다. 마치 '만약 ~하면, ~한다'는 약속이 코드로 작성되어 컴퓨터가 스스로 지키는 것과 같아요. 중간에 사람이 개입할 필요 없이 투명하고 안전하게 계약이 이행되도록 돕습니다.

**실무 예시:**
음료수 자판기처럼 돈을 넣으면 (조건) 음료수가 나오는 (실행) 방식과 비슷하죠. 이것이 확장되어 부동산 거래 시 잔금을 치르면 (조건) 자동으로 소유권이 이전되도록 (실행) 하는 등 다양한 ICT 분야에서 활용될 수 있습니다.


## Part 3. FastAPI로 웹 API 구조 만들기

실무에서는 모델을 직접 부르지 않고 서버를 거칩니다.
요청을 검증하고(`pydantic`), 응답 형식을 고정해 다른 시스템과 연결하기 쉽게 만듭니다.

Colab에서는 포트를 열지 않고 `TestClient`로 서버를 그대로 호출해 구조를 체험합니다.

In [34]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title='ICT Mini AI Agent')


class AskRequest(BaseModel):
    question: str = Field(..., min_length=2, max_length=300)


class AskResponse(BaseModel):
    answer: str
    model: str


@app.post('/ask', response_model=AskResponse)
def ask(request: AskRequest) -> AskResponse:
    api_key = os.getenv('GEMINI_API_KEY')
    if not api_key:
        raise HTTPException(status_code=500, detail='GEMINI_API_KEY가 설정되지 않았습니다.')
    answer = ask_gemini(request.question)
    return AskResponse(answer=answer, model=GEMINI_MODEL)


print('FastAPI 앱 정의 완료. 엔드포인트: POST /ask')

FastAPI 앱 정의 완료. 엔드포인트: POST /ask


In [35]:
from fastapi.testclient import TestClient

test_client = TestClient(app)

# 정상 요청
r = test_client.post('/ask', json={'question': 'API 키를 코드에 넣으면 안 되는 이유는?'})
print('상태코드:', r.status_code)
print(r.json())

상태코드: 200
{'answer': 'API 키를 코드에 직접 넣으면 안 되는 가장 큰 이유는 **보안 문제** 때문입니다.\n\n1.  코드가 외부에 노출될 경우(예: GitHub 같은 버전 관리 시스템), 다른 사람이 당신의 API 키를 쉽게 알아낼 수 있습니다.\n2.  이렇게 유출된 키는 악용되어 서비스에 무단으로 접근하거나, 당신의 계정으로 비용을 발생시키는 등의 심각한 문제가 생길 수 있습니다.\n3.  **실무 예시:** 만약 유료 지도 API 키를 코드에 넣어 GitHub에 올렸는데, 다른 사람이 이 키를 가져가 자기 앱에서 마음대로 사용해 버리면, 당신의 계정으로 수십만 원의 사용료가 청구될 수 있습니다.\n4.  따라서 API 키는 코드와 분리하여 환경 변수 등으로 따로 관리하는 것이 가장 안전합니다.', 'model': 'gemini-2.5-flash'}


In [36]:
# 잘못된 입력 (한 글자) → 422 검증 오류가 정상
bad = test_client.post('/ask', json={'question': 'a'})
print('상태코드:', bad.status_code, '(422면 입력 검증이 정상 작동)')

상태코드: 422 (422면 입력 검증이 정상 작동)


## Part 4. 과제 — 나만의 학습 도우미 만들기

`build_prompt`의 역할 설명을 바꿔 나만의 AI 도우미를 만드세요.

예시 역할: Python 문법 튜터 / 취업 면접 질문 생성기 / 네트워크 기초 설명 도우미 / 프로젝트 아이디어 추천

제출물: 수정한 프롬프트, 테스트 질문 3개와 응답, 실행 화면 캡처

In [37]:
def my_prompt(question: str) -> str:
    # TODO: 아래 역할 설명을 자신만의 도우미로 바꿔보세요
    return f'''너는 (여기에 역할을 적으세요. 예: Python 문법 튜터)다.
아래 질문에 친절하고 정확하게 답하라.

질문: {question}'''


def my_assistant(question: str) -> str:
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=my_prompt(question))
    return resp.text


# 테스트 질문 3개를 바꿔가며 실행해 보세요
for q in ['질문 1을 적으세요', '질문 2', '질문 3']:
    print('Q:', q)
    print('A:', my_assistant(q))
    print('-' * 40)

Q: 질문 1을 적으세요
A: 안녕하세요! 저에게 역할을 부여하고 질문을 주시려는 것이군요.

제가 어떤 역할을 맡기를 원하시는지 (예: Python 문법 튜터, 역사학자, 글쓰기 조언자 등) 구체적으로 알려주세요.

그리고 **'질문 1을 적으세요'** 부분에는 실제 질문 내용을 작성해 주세요.

예시처럼 역할과 질문을 명확히 알려주시면, 해당 역할에 맞춰 친절하고 정확하게 답변해 드리겠습니다.

준비되셨으면 다시 질문해 주세요!
----------------------------------------
Q: 질문 2
A: 안녕하세요! 질문해주셔서 감사합니다.

현재 제공해주신 형식에서 제가 어떤 역할을 맡아야 할지(예: Python 문법 튜터)와 구체적인 질문 내용이 비어있는 상태입니다.

제가 더욱 친절하고 정확하게 답변해 드릴 수 있도록, 다음 두 가지를 채워서 다시 질문해주시면 감사하겠습니다:

1.  **`(여기에 역할을 적으세요.)`** 부분에 제가 맡을 역할을 명확하게 지정해주세요.
    *   예: `Python 문법 튜터`, `여행 가이드`, `레시피 전문가`, `한국사 선생님` 등.
2.  **`질문: 질문 2`** 부분 대신에, 제가 답변해 드릴 **실제 질문 내용**을 구체적으로 작성해주세요.

**예시:**

---
너는 Python 문법 튜터다.
아래 질문에 친절하고 정확하게 답하라.

질문: Python에서 리스트(list)와 튜플(tuple)의 주요 차이점은 무엇인가요?
---

위 예시처럼 역할을 지정하고 실제 질문을 해주시면, 제가 해당 역할에 맞춰 최선을 다해 답변해 드리겠습니다! 😊
----------------------------------------
Q: 질문 3
A: 안녕하세요! 제가 어떤 역할을 맡아서 답변해 드려야 할지 알려주시면 감사하겠습니다.

예시처럼 'Python 문법 튜터'와 같은 구체적인 역할을 지정해 주시면, 그 역할에 맞춰 더 친절하고 정확하게 답변해 드릴 수 있습니다.

또한, '질문 3' 대신 